# 1 · Basic nodes

A node is a class. Its typed `run` signature is its interface: one `Annotated[DType, Param(...)]` per input, `Annotated[DType, Result(...)]` for the return. These notebooks borrow the standard types from `conductor_nodes.types`; a host declares its own.

In [ ]:
from dataclasses import dataclass
from typing import Annotated

from conductor import NodeRegistry, Param, Result, widgets, node
from conductor.widgets import Choice


In [ ]:
# instantiate a registry to hold our nodes.
registry = NodeRegistry()

# inspect our empty registry
registry

## One input, one output

One output is named `result`. `run` returns the type, never a bare `str`.

In [ ]:
from conductor import node
@node("greet", title="Greeting", description="Generates a greeting", category="text")
def greet(
    name: Annotated[str, Param(title="Name", description="Who to greet", widget=widgets.TextWidget())],
) -> Annotated[str, Result(title="Greeting")]:
    return f"Hello, {name}!"


registry = registry.extended_with([greet])

# A node is plain Python — instantiate it and call run directly:
greet(name="World")

In [ ]:
# inspect the stored node in the registry.
registry.nodes

## A dropdown

`Dropdown(choices=...)` stores the choice `id`. A default makes an input optional.

In [ ]:
from conductor import node
STYLES = (
    Choice(id="first-last", title="First Last"),
    Choice(id="last-first", title="Last, First"),
    Choice(id="upper-first", title="LAST First"),
)


@node("format-name", title="Format Name", description="Formats a full name", category="text")
def format_name(
    first: Annotated[str, Param(title="First name", widget=widgets.TextWidget())],
    last: Annotated[str, Param(title="Last name", widget=widgets.TextWidget())],
    style: Annotated[str, Param(title="Style", widget=widgets.Dropdown(choices=STYLES))] = "first-last",
) -> Annotated[str, Result(title="Full name")]:
    match style:
        case "last-first":
            return f"{last}, {first}"
        case "upper-first":
            return f"{last.upper()} {first}"
        case _:
            return f"{first} {last}"


registry = registry.extended_with([format_name])
format_name(first="Ada", last="Lovelace", style="upper-first")

## Slider, switch, textarea

In [ ]:
from conductor import node
@node("truncate", title="Truncate", description="Truncates text to a max length", category="text")
def truncate(
    text: Annotated[str, Param(title="Text", widget=widgets.Textarea(rows=3))],
    max_length: Annotated[
        float, Param(title="Max length", widget=widgets.Range(min_val=1, max_val=500, step=1))
    ] = 100.0,
    add_ellipsis: Annotated[bool, Param(title="Add ...", widget=widgets.Switch())] = True,
) -> Annotated[str, Result(title="Truncated")]:
    if len(text) <= max_length:
        return text
    truncated = text[: int(max_length)]
    return f"{truncated}..." if add_ellipsis else truncated


registry = registry.extended_with([truncate])
truncate(text="The quick brown fox jumps over the lazy dog", max_length=19.0)

## Several outputs

A frozen dataclass of `Result` fields. The field names are the output names.

In [ ]:
from conductor import node
@dataclass(frozen=True)
class Lines:
    first: Annotated[str, Result(title="First line")]
    rest: Annotated[str, Result(title="Remaining")]


@node("split-text", title="Split Text", description="Splits text into first line and rest", category="text")
def split_text(text: Annotated[str, Param(title="Text", widget=widgets.Textarea(rows=4))]) -> Lines:
    first, _, rest = text.partition("\n")
    return Lines(first=first, rest=rest)


registry = registry.extended_with([split_text])
split_text(text="hello\nworld\nagain")

## The registry

Each class carries its versions; each version its `Interface`, the records a frontend renders.

In [ ]:
for cls in registry.nodes:
    interface = cls.versions[1].interface
    inputs = ", ".join(f"{i.name}: {i.dtype.id}" for i in interface.inputs)
    outputs = ", ".join(f"{o.name}: {o.dtype.id}" for o in interface.outputs)
    print(f"  {cls.id}: {cls.title}")
    print(f"    inputs:  ({inputs})")
    print(f"    outputs: ({outputs})")
    print()

registry

## The standard nodes

`conductor-nodes` ships text, math, logic, JSON and regex nodes and a `decision` gate:

```python
import conductor_nodes

registry = conductor_nodes.registry(categories=["text", "math"])
conductor_nodes.register_all(my_registry, categories=["logic"])
```